# 05 - Build `ml.trip_validity_dataset`

The final, ML-ready table for the Trip Validity model: every identifier,
every raw metric from `ml.trip_validity_trip_metrics`, and every
relative/normalized column surveyed and agreed on beforehand - 80
columns total (15 identifiers + 40 existing metrics/counts + 25 new
relative columns).

Built with a single `CREATE TABLE ... AS SELECT` from
`ml.trip_validity_trip_metrics` joined to `ml.trip_validity_trips` (for
`bus_id`/`trip_date`/`trip_hour`/`trip_opening_timestamp`/
`trip_closing_timestamp`, which live on `trips` but weren't carried onto
`trip_metrics`) and left-joined to `ml.trip_validity_bus_avl_match`
(built in `04_avl_positions.ipynb`, for `avl_matched`/`avl_match_source`)
- no expensive spatial recomputation, just arithmetic on already-materialized
columns plus two cheap key lookups, so this should run in seconds, not
minutes.

**Column order**: identifiers first, then five themed groups (duration,
fares, geometry, path match, directional progress), each ordered raw →
count → relative, so a raw value and the ratio computed from it always
sit next to each other.

**Two conventions applied uniformly**:
- Every ratio divides through `NULLIF(denominator, 0)`, since Postgres's
  floating-point division does not error on divide-by-zero - it silently
  returns `Infinity`, which would sit in the table looking like a real
  number and quietly break anything downstream.
- The four `path_match_score_*` columns are clipped to `[0, 1]` via
  `GREATEST(0, ...)`- but `GREATEST`/`LEAST` in Postgres **ignore NULL
  arguments** rather than propagating them (unlike plain arithmetic,
  which is NULL-safe), so `GREATEST(0, NULL)` silently returns `0`, not
  `NULL`. Left unguarded, a trip with no matched shape would wrongly
  show a match score of `0` ("worst possible match") instead of `NULL`
  ("not applicable"). Each of those four columns is wrapped in a `CASE`
  that checks whether the *unclipped* ratio is `NULL` first, and only
  applies `GREATEST` when it's a real number.

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

connected


## Build the table

In [4]:
conn.execute("DROP TABLE IF EXISTS ml.trip_validity_dataset CASCADE;")

conn.execute("""
    CREATE TABLE ml.trip_validity_dataset AS
    SELECT
        -- ===== Identifiers =====
        tm.trip_id,
        t.bus_id,
        tm.route_id,
        tm.route_direction,
        t.trip_date,
        t.trip_hour,
        t.trip_opening_timestamp,
        t.trip_closing_timestamp,
        tm.gtfs_feed_version_date,
        tm.gtfs_route_short_name,
        tm.gtfs_shape_id_i,
        tm.gtfs_shape_id_v,
        tm.gtfs_route_has_both_directions,

        -- ===== AVL position matching =====
        COALESCE(bm.avl_matched, false) AS avl_matched,
        bm.avl_match_source,

        -- ===== Duration vs. expectations =====
        tm.trip_duration_seconds,

        tm.route_avg_trip_duration_seconds_loo,
        tm.route_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_avg,

        tm.route_direction_avg_trip_duration_seconds_loo,
        tm.route_direction_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_direction_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_direction_avg,

        tm.route_hour_avg_trip_duration_seconds_loo,
        tm.route_hour_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_hour_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_hour_avg,

        tm.route_direction_hour_avg_trip_duration_seconds_loo,
        tm.route_direction_hour_avg_trip_duration_n_trips_loo,
        tm.trip_duration_seconds
            / NULLIF(tm.route_direction_hour_avg_trip_duration_seconds_loo, 0)
            AS trip_duration_ratio_to_route_direction_hour_avg,

        tm.route_reverse_direction_avg_trip_duration_seconds,
        tm.route_reverse_direction_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_reverse_direction_avg_trip_duration_seconds, 0)
            AS trip_duration_ratio_to_reverse_direction_avg,

        tm.route_reverse_direction_hour_avg_trip_duration_seconds,
        tm.route_reverse_direction_hour_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_reverse_direction_hour_avg_trip_duration_seconds, 0)
            AS trip_duration_ratio_to_reverse_direction_hour_avg,

        tm.route_i_scheduled_duration_avg_seconds,
        tm.route_i_scheduled_duration_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_i_scheduled_duration_avg_seconds, 0)
            AS trip_duration_ratio_to_scheduled_i,

        tm.route_v_scheduled_duration_avg_seconds,
        tm.route_v_scheduled_duration_n_trips,
        tm.trip_duration_seconds
            / NULLIF(tm.route_v_scheduled_duration_avg_seconds, 0)
            AS trip_duration_ratio_to_scheduled_v,

        tm.route_i_scheduled_duration_avg_seconds_at_hour,
        tm.route_i_scheduled_duration_n_trips_at_hour,
        tm.trip_duration_seconds
            / NULLIF(tm.route_i_scheduled_duration_avg_seconds_at_hour, 0)
            AS trip_duration_ratio_to_scheduled_i_at_hour,

        tm.route_v_scheduled_duration_avg_seconds_at_hour,
        tm.route_v_scheduled_duration_n_trips_at_hour,
        tm.trip_duration_seconds
            / NULLIF(tm.route_v_scheduled_duration_avg_seconds_at_hour, 0)
            AS trip_duration_ratio_to_scheduled_v_at_hour,

        -- ===== Fare activity =====
        tm.trip_fare_count,
        tm.fare_gap_avg_seconds,
        tm.fare_gap_stddev_seconds,
        tm.fare_gap_stddev_seconds / NULLIF(tm.fare_gap_avg_seconds, 0)
            AS fare_gap_coefficient_of_variation,
        tm.fare_gap_avg_seconds / NULLIF(tm.trip_duration_seconds, 0)
            AS fare_gap_avg_ratio_to_duration,
        tm.fare_span_seconds,
        tm.fare_span_seconds / NULLIF(tm.trip_duration_seconds, 0)
            AS fare_span_ratio_to_duration,

        -- ===== Trip geometry vs. route length =====
        tm.trip_distance_meters,
        tm.route_i_length_meters,
        tm.trip_distance_meters / NULLIF(tm.route_i_length_meters, 0)
            AS trip_distance_ratio_to_route_i,
        tm.route_v_length_meters,
        tm.trip_distance_meters / NULLIF(tm.route_v_length_meters, 0)
            AS trip_distance_ratio_to_route_v,
        tm.trip_points_standard_distance_meters,
        tm.trip_points_standard_distance_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_cohesion_ratio_to_route_i,
        tm.trip_points_standard_distance_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_cohesion_ratio_to_route_v,

        -- ===== Path shape match =====
        tm.trip_start_distance_to_i_start_meters,
        tm.trip_start_distance_to_i_start_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_start_offset_ratio_to_i,
        tm.trip_start_distance_to_v_start_meters,
        tm.trip_start_distance_to_v_start_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_start_offset_ratio_to_v,
        tm.trip_end_distance_to_i_end_meters,
        tm.trip_end_distance_to_i_end_meters
            / NULLIF(tm.route_i_length_meters, 0)
            AS trip_end_offset_ratio_to_i,
        tm.trip_end_distance_to_v_end_meters,
        tm.trip_end_distance_to_v_end_meters
            / NULLIF(tm.route_v_length_meters, 0)
            AS trip_end_offset_ratio_to_v,

        tm.path_frechet_distance_to_i_meters,
        CASE WHEN 1 - tm.path_frechet_distance_to_i_meters
                      / NULLIF(tm.route_i_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_frechet_distance_to_i_meters
                                  / NULLIF(tm.route_i_length_meters, 0))
        END AS path_match_score_frechet_i,

        tm.path_frechet_distance_to_v_meters,
        CASE WHEN 1 - tm.path_frechet_distance_to_v_meters
                      / NULLIF(tm.route_v_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_frechet_distance_to_v_meters
                                  / NULLIF(tm.route_v_length_meters, 0))
        END AS path_match_score_frechet_v,

        tm.path_hausdorff_distance_to_i_meters,
        CASE WHEN 1 - tm.path_hausdorff_distance_to_i_meters
                      / NULLIF(tm.route_i_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_hausdorff_distance_to_i_meters
                                  / NULLIF(tm.route_i_length_meters, 0))
        END AS path_match_score_hausdorff_i,

        tm.path_hausdorff_distance_to_v_meters,
        CASE WHEN 1 - tm.path_hausdorff_distance_to_v_meters
                      / NULLIF(tm.route_v_length_meters, 0) IS NULL
             THEN NULL
             ELSE GREATEST(0, 1 - tm.path_hausdorff_distance_to_v_meters
                                  / NULLIF(tm.route_v_length_meters, 0))
        END AS path_match_score_hausdorff_v,

        -- ===== Directional progress =====
        tm.trip_progress_correlation_to_i,
        tm.trip_progress_correlation_to_v,
        tm.trip_progress_correlation_n_points

    FROM ml.trip_validity_trip_metrics tm
    JOIN ml.trip_validity_trips t ON t.trip_id = tm.trip_id
    LEFT JOIN ml.trip_validity_bus_avl_match bm ON bm.bus_id = t.bus_id
    ORDER BY tm.trip_id;
""")
conn.commit()

EXPECTED_TRIP_COUNT = 940988
EXPECTED_COLUMN_COUNT = 80

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_dataset;")
    row_count = cur.fetchone()[0]
    print("rows:", row_count)
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)

if row_count != EXPECTED_TRIP_COUNT:
    msg = "row count mismatch against trip_validity_trips"
    raise AssertionError(msg)
if column_count != EXPECTED_COLUMN_COUNT:
    msg = "column count mismatch against the plan"
    raise AssertionError(msg)

rows: 940988
columns: 80


## Constraints and indexes

`CREATE TABLE ... AS SELECT` doesn't carry over constraints, so adding
them explicitly: the primary key (and FK back to `trip_validity_trips`,
for traceability), `NOT NULL` on the columns known to always be
populated (mirroring their source tables' own guarantees), and indexes
on the columns most likely to be used for filtering/sampling during
active learning.

In [5]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD PRIMARY KEY (trip_id),
        ADD FOREIGN KEY (trip_id) REFERENCES ml.trip_validity_trips (trip_id),
        ALTER COLUMN bus_id SET NOT NULL,
        ALTER COLUMN route_id SET NOT NULL,
        ALTER COLUMN route_direction SET NOT NULL,
        ALTER COLUMN trip_date SET NOT NULL,
        ALTER COLUMN trip_hour SET NOT NULL,
        ALTER COLUMN trip_opening_timestamp SET NOT NULL,
        ALTER COLUMN trip_closing_timestamp SET NOT NULL,
        ALTER COLUMN avl_matched SET NOT NULL,
        ALTER COLUMN trip_fare_count SET NOT NULL;
""")

conn.execute("""
    CREATE INDEX trip_validity_dataset_route_id_idx
        ON ml.trip_validity_dataset (route_id);
    CREATE INDEX trip_validity_dataset_route_direction_idx
        ON ml.trip_validity_dataset (route_id, route_direction);
    CREATE INDEX trip_validity_dataset_bus_id_idx
        ON ml.trip_validity_dataset (bus_id);
    CREATE INDEX trip_validity_dataset_trip_date_idx
        ON ml.trip_validity_dataset (trip_date);
    ANALYZE ml.trip_validity_dataset;
""")
conn.commit()
print("constraints and indexes applied")

constraints and indexes applied


## Column-level provenance comments

In [6]:
def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


NULLIF_NOTE = (
    " Ratio divides through NULLIF(denominator, 0), so a zero denominator "
    "gives NULL rather than the Infinity Postgres would otherwise "
    "silently produce."
)
CLIP_NOTE = (
    " Clipped to [0, 1] via GREATEST(0, ...), NULL-safely (unlike bare "
    "GREATEST, which would turn a NULL input into 0)."
)

DATASET_COMMENTS = {
    # Identifiers
    "trip_id": (
        "PK. = ml.trip_validity_trips.trip_id / ml.trip_validity_trip_metrics.trip_id."
    ),
    "bus_id": "From ml.trip_validity_trips.bus_id.",
    "route_id": "From ml.trip_validity_trip_metrics.route_id.",
    "route_direction": (
        "From ml.trip_validity_trip_metrics.route_direction (this trip's "
        "own observed AFC direction)."
    ),
    "trip_date": "From ml.trip_validity_trips.trip_date.",
    "trip_hour": "From ml.trip_validity_trips.trip_hour.",
    "trip_opening_timestamp": (
        "From ml.trip_validity_trips.trip_opening_timestamp (UTC). Start "
        "of the AVL position window used to build "
        "ml.trip_validity_trip_positions."
    ),
    "trip_closing_timestamp": (
        "From ml.trip_validity_trips.trip_closing_timestamp (UTC). End of "
        "the AVL position window. May be the 1899-12-30 Delphi zero-date "
        "sentinel (see trip_duration_seconds) - such trips have no "
        "positions, not garbage ones, since closing < opening yields an "
        "empty window."
    ),
    "gtfs_feed_version_date": (
        "From ml.trip_validity_trip_metrics (originally "
        "ml.trip_validity_route_gtfs_match)."
    ),
    "gtfs_route_short_name": (
        "From ml.trip_validity_trip_metrics - the GTFS join key actually used."
    ),
    "gtfs_shape_id_i": "From ml.trip_validity_trip_metrics.",
    "gtfs_shape_id_v": "From ml.trip_validity_trip_metrics.",
    "gtfs_route_has_both_directions": (
        "From ml.trip_validity_trip_metrics. NULL = route never matched "
        "any GTFS feed; false = matched but only one direction has a "
        "shape; true = both directions have a shape."
    ),
    "avl_matched": (
        "From ml.trip_validity_bus_avl_match, keyed by bus_id (not "
        "trip_id - every trip on the same bus shares the same match). "
        "True iff this trip's bus_id was resolved to an AVL vehicle_id "
        "or device_id, i.e. ml.trip_validity_trip_positions can have rows "
        "for this trip. False (never NULL) otherwise."
    ),
    "avl_match_source": (
        "Which silver crosswalk resolved the match: "
        "'dictionary_device' (silver.dictionary_device.codigo -> "
        "device_id, preferred when both match) or 'dictionary_vehicle' "
        "(silver.dictionary_vehicle.cod_veiculo -> id_veiculo, used only "
        "when no dictionary_device match exists). NULL iff avl_matched "
        "is false."
    ),
    # Duration
    "trip_duration_seconds": (
        "From ml.trip_validity_trip_metrics. NULL for the 9 trips with "
        "the 1899-12-30 Delphi zero-date sentinel closing time."
    ),
    "route_avg_trip_duration_seconds_loo": (
        "Leave-one-out mean duration across other trips on this route."
    ),
    "route_avg_trip_duration_n_trips_loo": "Count backing the average above.",
    "trip_duration_ratio_to_route_avg": (
        "trip_duration_seconds / route_avg_trip_duration_seconds_loo. "
        "1.0 = exactly average; >1 = slower; <1 = faster." + NULLIF_NOTE
    ),
    "route_direction_avg_trip_duration_seconds_loo": (
        "Same as route_avg_trip_duration_seconds_loo, grouped by "
        "(route_id, route_direction)."
    ),
    "route_direction_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "trip_duration_ratio_to_route_direction_avg": (
        "trip_duration_seconds / "
        "route_direction_avg_trip_duration_seconds_loo." + NULLIF_NOTE
    ),
    "route_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, trip_hour)."
    ),
    "route_hour_avg_trip_duration_n_trips_loo": "Count backing the average above.",
    "trip_duration_ratio_to_route_hour_avg": (
        "trip_duration_seconds / route_hour_avg_trip_duration_seconds_loo."
        + NULLIF_NOTE
    ),
    "route_direction_hour_avg_trip_duration_seconds_loo": (
        "Same, grouped by (route_id, route_direction, trip_hour)."
    ),
    "route_direction_hour_avg_trip_duration_n_trips_loo": (
        "Count backing the average above."
    ),
    "trip_duration_ratio_to_route_direction_hour_avg": (
        "trip_duration_seconds / "
        "route_direction_hour_avg_trip_duration_seconds_loo." + NULLIF_NOTE
    ),
    "route_reverse_direction_avg_trip_duration_seconds": (
        "Mean duration of trips on this route going the OPPOSITE "
        "route_direction. Not leave-one-out (this trip can't be a "
        "member of that group)."
    ),
    "route_reverse_direction_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_reverse_direction_avg": (
        "trip_duration_seconds / "
        "route_reverse_direction_avg_trip_duration_seconds. How this "
        "trip compares to the OTHER direction's typical duration." + NULLIF_NOTE
    ),
    "route_reverse_direction_hour_avg_trip_duration_seconds": (
        "Same as route_reverse_direction_avg_trip_duration_seconds, "
        "restricted to the opposite direction's trips at this row's "
        "trip_hour."
    ),
    "route_reverse_direction_hour_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_reverse_direction_hour_avg": (
        "trip_duration_seconds / "
        "route_reverse_direction_hour_avg_trip_duration_seconds." + NULLIF_NOTE
    ),
    "route_i_scheduled_duration_avg_seconds": (
        "GTFS-scheduled average duration for direction I on this "
        "route/feed (from silver.gtfs_stop_times, via "
        "ml.trip_validity_route_schedule). What the TIMETABLE says, not "
        "other real trips."
    ),
    "route_i_scheduled_duration_n_trips": (
        "Count of scheduled GTFS trips backing the average above."
    ),
    "trip_duration_ratio_to_scheduled_i": (
        "trip_duration_seconds / route_i_scheduled_duration_avg_seconds. "
        "Actual vs. planned." + NULLIF_NOTE
    ),
    "route_v_scheduled_duration_avg_seconds": (
        "Same as route_i_scheduled_duration_avg_seconds, direction V."
    ),
    "route_v_scheduled_duration_n_trips": "Count backing the average above.",
    "trip_duration_ratio_to_scheduled_v": (
        "trip_duration_seconds / route_v_scheduled_duration_avg_seconds." + NULLIF_NOTE
    ),
    "route_i_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds, restricted to "
        "scheduled GTFS trips whose start_hour matches this row's "
        "trip_hour."
    ),
    "route_i_scheduled_duration_n_trips_at_hour": ("Count backing the average above."),
    "trip_duration_ratio_to_scheduled_i_at_hour": (
        "trip_duration_seconds / "
        "route_i_scheduled_duration_avg_seconds_at_hour. Actual vs. "
        "planned, hour-matched." + NULLIF_NOTE
    ),
    "route_v_scheduled_duration_avg_seconds_at_hour": (
        "Same as route_i_scheduled_duration_avg_seconds_at_hour, direction V."
    ),
    "route_v_scheduled_duration_n_trips_at_hour": ("Count backing the average above."),
    "trip_duration_ratio_to_scheduled_v_at_hour": (
        "trip_duration_seconds / "
        "route_v_scheduled_duration_avg_seconds_at_hour." + NULLIF_NOTE
    ),
    # Fares
    "trip_fare_count": (
        "From ml.trip_validity_trip_metrics. count(*) of all fare taps "
        "on this trip, geo-tagged or not."
    ),
    "fare_gap_avg_seconds": (
        "Mean gap between consecutive fare taps, ordered by boarding_at. "
        "NULL if only 1 fare."
    ),
    "fare_gap_stddev_seconds": (
        "Sample stddev of the same gaps. NULL if fewer than 2 gaps exist."
    ),
    "fare_gap_coefficient_of_variation": (
        "fare_gap_stddev_seconds / fare_gap_avg_seconds. Standard "
        "normalized-dispersion measure: were the gaps steady or erratic, "
        "independent of trip length." + NULLIF_NOTE
    ),
    "fare_gap_avg_ratio_to_duration": (
        "fare_gap_avg_seconds / trip_duration_seconds. What fraction of "
        "the whole trip is 'typical time between taps'." + NULLIF_NOTE
    ),
    "fare_span_seconds": (
        "max(boarding_at) - min(boarding_at) across all fares. 0 (not "
        "NULL) for a 1-fare trip."
    ),
    "fare_span_ratio_to_duration": (
        "fare_span_seconds / trip_duration_seconds. What fraction of "
        "the trip's duration actually had fare activity." + NULLIF_NOTE
    ),
    # Geometry
    "trip_distance_meters": (
        "ST_Length(trip_path::geography). NULL if trip_path is NULL "
        "(fewer than 2 geo-tagged fares)."
    ),
    "route_i_length_meters": (
        "Official route length, direction I (ml.trip_validity_route_shapes)."
    ),
    "trip_distance_ratio_to_route_i": (
        "trip_distance_meters / route_i_length_meters. Did the GPS trace "
        "cover roughly the whole route, or just a fraction of it?" + NULLIF_NOTE
    ),
    "route_v_length_meters": "Official route length, direction V.",
    "trip_distance_ratio_to_route_v": (
        "trip_distance_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "trip_points_standard_distance_meters": (
        "RMS distance of this trip's geo-tagged fares from their own "
        "centroid (spatial-statistics standard distance). 0 for a "
        "single point, NULL for zero."
    ),
    "trip_cohesion_ratio_to_route_i": (
        "trip_points_standard_distance_meters / route_i_length_meters. "
        "Is the point spread large or small relative to how long this "
        "route actually is?" + NULLIF_NOTE
    ),
    "trip_cohesion_ratio_to_route_v": (
        "trip_points_standard_distance_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    # Path match
    "trip_start_distance_to_i_start_meters": (
        "ST_Distance (geography) between the trip's first GPS point and "
        "the official I-direction route's start point."
    ),
    "trip_start_offset_ratio_to_i": (
        "trip_start_distance_to_i_start_meters / route_i_length_meters." + NULLIF_NOTE
    ),
    "trip_start_distance_to_v_start_meters": (
        "Same as trip_start_distance_to_i_start_meters, direction V."
    ),
    "trip_start_offset_ratio_to_v": (
        "trip_start_distance_to_v_start_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "trip_end_distance_to_i_end_meters": (
        "ST_Distance (geography) between the trip's last GPS point and "
        "the official I-direction route's end point."
    ),
    "trip_end_offset_ratio_to_i": (
        "trip_end_distance_to_i_end_meters / route_i_length_meters." + NULLIF_NOTE
    ),
    "trip_end_distance_to_v_end_meters": (
        "Same as trip_end_distance_to_i_end_meters, direction V."
    ),
    "trip_end_offset_ratio_to_v": (
        "trip_end_distance_to_v_end_meters / route_v_length_meters." + NULLIF_NOTE
    ),
    "path_frechet_distance_to_i_meters": (
        "ST_FrechetDistance between the trip's path and the I-direction "
        "shape (both projected to EPSG:31984 first). Order-aware path "
        "similarity distance, meters."
    ),
    "path_match_score_frechet_i": (
        "1 - (path_frechet_distance_to_i_meters / route_i_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_frechet_distance_to_v_meters": (
        "Same as path_frechet_distance_to_i_meters, direction V."
    ),
    "path_match_score_frechet_v": (
        "1 - (path_frechet_distance_to_v_meters / route_v_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_hausdorff_distance_to_i_meters": (
        "ST_HausdorffDistance between the trip's path and the "
        "I-direction shape (both projected to EPSG:31984 first). "
        "Order-agnostic 'largest gap' path similarity distance, meters."
    ),
    "path_match_score_hausdorff_i": (
        "1 - (path_hausdorff_distance_to_i_meters / route_i_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    "path_hausdorff_distance_to_v_meters": (
        "Same as path_hausdorff_distance_to_i_meters, direction V."
    ),
    "path_match_score_hausdorff_v": (
        "1 - (path_hausdorff_distance_to_v_meters / route_v_length_meters)."
        + NULLIF_NOTE
        + CLIP_NOTE
    ),
    # Directional progress
    "trip_progress_correlation_to_i": (
        "Pearson correlation between position-along-the-I-shape "
        "(ST_LineLocatePoint) and boarding_at, across this trip's "
        "geo-tagged fares. +1 = steady forward progress, -1 = steady "
        "reverse progress, ~0 = no consistent relationship. Already "
        "bounded [-1, 1] by construction - not further normalized."
    ),
    "trip_progress_correlation_to_v": (
        "Same as trip_progress_correlation_to_i, direction V."
    ),
    "trip_progress_correlation_n_points": (
        "Count of geo-tagged fares used for both correlations above. A "
        "value of 2 means the correlation is a meaningless exact +-1 (a "
        "line through 2 points is always 'perfectly correlated') - use "
        "this to judge how much to trust the two correlation columns."
    ),
}

with conn.cursor() as cur:
    for col, text in DATASET_COMMENTS.items():
        comment_on_column(cur, "trip_validity_dataset", col, text)

conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_dataset IS {};").format(
        sql.Literal(
            "Trip Validity model: the final ML-ready dataset. One row per "
            "trip, 80 columns (identifiers + raw metrics from "
            "ml.trip_validity_trip_metrics + every relative/normalized "
            "column). See ml/trip_validity_model/notebooks/"
            "05_final_dataset.ipynb."
        )
    )
)
conn.commit()
print(f"comments applied for {len(DATASET_COMMENTS)} columns")

comments applied for 80 columns


## Verification

Row/column counts, a check that no ratio silently produced `Infinity`
(would mean a `NULLIF` guard was missed somewhere), and a hand-check of
one duration ratio against its raw inputs.

In [7]:
RATIO_COLUMNS = [
    "trip_duration_ratio_to_route_avg",
    "trip_duration_ratio_to_route_direction_avg",
    "trip_duration_ratio_to_route_hour_avg",
    "trip_duration_ratio_to_route_direction_hour_avg",
    "trip_duration_ratio_to_reverse_direction_avg",
    "trip_duration_ratio_to_reverse_direction_hour_avg",
    "trip_duration_ratio_to_scheduled_i",
    "trip_duration_ratio_to_scheduled_v",
    "trip_duration_ratio_to_scheduled_i_at_hour",
    "trip_duration_ratio_to_scheduled_v_at_hour",
    "fare_gap_coefficient_of_variation",
    "fare_gap_avg_ratio_to_duration",
    "fare_span_ratio_to_duration",
    "trip_distance_ratio_to_route_i",
    "trip_distance_ratio_to_route_v",
    "trip_cohesion_ratio_to_route_i",
    "trip_cohesion_ratio_to_route_v",
    "trip_start_offset_ratio_to_i",
    "trip_start_offset_ratio_to_v",
    "trip_end_offset_ratio_to_i",
    "trip_end_offset_ratio_to_v",
    "path_match_score_frechet_i",
    "path_match_score_frechet_v",
    "path_match_score_hausdorff_i",
    "path_match_score_hausdorff_v",
]
EXPECTED_RATIO_COLUMN_COUNT = 25
FLOAT_TOLERANCE = 1e-9

if len(RATIO_COLUMNS) != EXPECTED_RATIO_COLUMN_COUNT:
    msg = (
        f"expected {EXPECTED_RATIO_COLUMN_COUNT} new ratio columns, "
        f"listed {len(RATIO_COLUMNS)}"
    )
    raise AssertionError(msg)

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.trip_validity_dataset;")
    print("rows:", cur.fetchone()[0])

    # no silent Infinity anywhere - built via safe sql.Identifier
    # composition, not raw string interpolation, even though
    # RATIO_COLUMNS is a fixed, hardcoded list (not user input)
    infinity_check = sql.SQL(" + ").join(
        sql.SQL(
            "count(*) FILTER (WHERE {0} = 'Infinity'::double precision "
            "OR {0} = '-Infinity'::double precision)"
        ).format(sql.Identifier(c))
        for c in RATIO_COLUMNS
    )
    cur.execute(
        sql.SQL("SELECT {} FROM ml.trip_validity_dataset;").format(infinity_check)
    )
    n_infinite = cur.fetchone()[0]
    print("rows with an Infinity value across all 25 ratio columns:", n_infinite)
    if n_infinite != 0:
        msg = "found Infinity - a NULLIF guard was missed"
        raise AssertionError(msg)

    # path_match_score stays in [0, 1] wherever it's not NULL
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE (path_match_score_frechet_i IS NOT NULL
               AND (path_match_score_frechet_i < 0
                    OR path_match_score_frechet_i > 1))
           OR (path_match_score_frechet_v IS NOT NULL
               AND (path_match_score_frechet_v < 0
                    OR path_match_score_frechet_v > 1))
           OR (path_match_score_hausdorff_i IS NOT NULL
               AND (path_match_score_hausdorff_i < 0
                    OR path_match_score_hausdorff_i > 1))
           OR (path_match_score_hausdorff_v IS NOT NULL
               AND (path_match_score_hausdorff_v < 0
                    OR path_match_score_hausdorff_v > 1));
    """)
    out_of_range = cur.fetchone()[0]
    print("path_match_score rows out of [0,1]:", out_of_range)
    if out_of_range != 0:
        msg = "path_match_score found out of [0,1] range"
        raise AssertionError(msg)

    # NULL-safety check: path_match_score must be NULL exactly when
    # frechet_i is NULL, never a stray 0
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE path_frechet_distance_to_i_meters IS NULL
          AND path_match_score_frechet_i IS NOT NULL;
    """)
    leaked_zero = cur.fetchone()[0]
    print(
        "rows where a NULL frechet distance wrongly produced a non-null score:",
        leaked_zero,
    )
    if leaked_zero != 0:
        msg = "a NULL frechet distance wrongly produced a non-null score"
        raise AssertionError(msg)

    # hand-check one ratio against its raw inputs, on a real row
    cur.execute("""
        SELECT trip_id, trip_duration_seconds,
               route_avg_trip_duration_seconds_loo,
               trip_duration_ratio_to_route_avg
        FROM ml.trip_validity_dataset
        WHERE trip_duration_ratio_to_route_avg IS NOT NULL
        ORDER BY trip_id LIMIT 1;
    """)
    trip_id, dur, avg, ratio = cur.fetchone()
    print(
        f"hand-check trip {trip_id}: {dur} / {avg} = {dur / avg:.6f}  "
        f"(stored: {ratio:.6f})"
    )
    if abs(dur / avg - ratio) >= FLOAT_TOLERANCE:
        msg = "hand-check ratio does not match stored value"
        raise AssertionError(msg)

print("all checks passed")

rows: 940988


rows with an Infinity value across all 25 ratio columns: 0
path_match_score rows out of [0,1]: 0


rows where a NULL frechet distance wrongly produced a non-null score: 0
hand-check trip 1: 1158 / 1545.5891959798994 = 0.749229  (stored: 0.749229)
all checks passed


## Querying: GTFS route shape + AVL positions for a trip

Both joins are PK lookups (confirmed ~5ms/~0ms via `EXPLAIN ANALYZE`),
so this stays fast regardless of table size - no need to touch
`silver` or scan anything:

```sql
SELECT
    d.trip_id, d.bus_id, d.route_id, d.avl_matched,
    rs.shape_geom AS gtfs_route_geom,   -- official route line (LineString)
    pos.metric_timestamp, pos.geom AS bus_position
FROM ml.trip_validity_dataset d
LEFT JOIN ml.trip_validity_route_shapes rs
    ON rs.feed_version_date = d.gtfs_feed_version_date
   AND rs.shape_id = COALESCE(d.gtfs_shape_id_i, d.gtfs_shape_id_v)
LEFT JOIN ml.trip_validity_trip_positions pos
    ON pos.trip_id = d.trip_id
WHERE d.trip_id = 12345
ORDER BY pos.metric_timestamp;
```

Both are `LEFT JOIN`s on purpose: `rs` can be empty (~30 routes never
matched any GTFS feed) and `pos` can be empty (`avl_matched = false`,
or matched but no pings actually fell in the trip's window) - check
`gtfs_feed_version_date`/`avl_matched` if you need to tell "no route" /
"no positions" apart from "not queried yet". `COALESCE(..._i, ..._v)`
picks a default when `gtfs_route_has_both_directions` is true; use
`path_match_score_frechet_i`/`_v` (already computed) to pick whichever
shape this trip's *own* GPS path actually matches better, instead.

## Adding weekday/weekend features

For the Trip Validity active-learning model: `weekday_number` and
`is_weekend`, both derived from the already-present `trip_date` via
`GENERATED ALWAYS AS ... STORED` - the same pattern this project
already uses for the PostGIS geometry columns in silver. Postgres
computes the value for every existing row as part of the `ALTER
TABLE` itself, so there's no separate backfill/update step, and the
columns can never drift out of sync with `trip_date` since Postgres
recomputes them automatically on any (hypothetical, trip_date is
otherwise never updated) change.

`weekday_number` uses ISO 8601 numbering (`EXTRACT(ISODOW FROM
trip_date)`: 1 = Monday ... 7 = Sunday) rather than Postgres's default
`DOW` (0 = Sunday ... 6 = Saturday), to avoid the Sunday/Monday-as-0
ambiguity. `is_weekend` is `true` for Saturday/Sunday (6, 7).

In [8]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD COLUMN IF NOT EXISTS weekday_number SMALLINT
            GENERATED ALWAYS AS (EXTRACT(ISODOW FROM trip_date)::smallint) STORED,
        ADD COLUMN IF NOT EXISTS is_weekend BOOLEAN
            GENERATED ALWAYS AS (EXTRACT(ISODOW FROM trip_date) IN (6, 7)) STORED;
""")

conn.execute(
    sql.SQL("COMMENT ON COLUMN ml.trip_validity_dataset.weekday_number IS {};").format(
        sql.Literal(
            "GENERATED ALWAYS AS (EXTRACT(ISODOW FROM trip_date)::smallint) STORED. "
            "ISO 8601 weekday: 1 = Monday ... 7 = Sunday (not Postgres's default "
            "DOW, which is 0 = Sunday)."
        )
    )
)
conn.execute(
    sql.SQL("COMMENT ON COLUMN ml.trip_validity_dataset.is_weekend IS {};").format(
        sql.Literal(
            "GENERATED ALWAYS AS (EXTRACT(ISODOW FROM trip_date) IN (6, 7)) STORED. "
            "True for Saturday/Sunday."
        )
    )
)
conn.commit()
print("weekday_number/is_weekend added")

weekday_number/is_weekend added


In [9]:
EXPECTED_COLUMN_COUNT_AFTER_WEEKDAY = 82

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)
    if column_count != EXPECTED_COLUMN_COUNT_AFTER_WEEKDAY:
        msg = "column count mismatch after adding weekday_number/is_weekend"
        raise AssertionError(msg)

    cur.execute("""
        SELECT trip_date, weekday_number, is_weekend
        FROM ml.trip_validity_dataset ORDER BY trip_id LIMIT 3;
    """)
    for row in cur.fetchall():
        print(row)

print("all checks passed")

columns: 82
(datetime.date(2023, 11, 9), 4, False)
(datetime.date(2023, 11, 10), 5, False)
(datetime.date(2023, 11, 9), 4, False)
all checks passed


## Adding garage-distance and company_id features

Two more engineered features for the Trip Validity model: per trip, the
distance from the trip's **first** and **last geo-tagged AFC fare tap**
(never AVL - only what a rider's card actually tapped, at a real lat/lon)
to the **nearest bus garage of that trip's operating company**. Several
companies run more than one garage (Vega: Jacarecanga + Messejana;
COOTRAPS: Autran Nunes + Jangurussu; Santa Maria has three nearby points
on file), so "nearest" takes `MIN()` across every known garage for that
company, not just its registered address.

Garage coordinates are **hard-coded** below (`GARAGES`), taken from
manually verified Google Maps pins (preferred over this project's own
OSM/Nominatim geocoding of each company's registered postal address,
which for two companies was off by 1.9-3 km because the address didn't
resolve to an exact building in OSM). There's no silver/bronze table for
this - it's external reference data with no upstream feed - so unlike
everything else in this pipeline it can't be reloaded from source; if a
company opens or closes a garage, `GARAGES` needs a manual edit.

A trip's first/last geo-tagged fare are read straight from
`ml.trip_validity_trip_fares` (`geom IS NOT NULL`, ordered by
`boarding_at`) rather than from `trip_path`: `trip_path` is `NULL` for
any trip with fewer than 2 geo-tagged fares (see
`01_build_trip_tables.ipynb`), which would silently drop every
single-tap trip's distance instead of just collapsing its start/end
distance to the same value.

`company_id` (the 2-digit `bus_id` prefix, e.g. `"02"` - leading zero
significant, it's the operator's registry code, not a number) is a
`GENERATED ALWAYS AS (LEFT(bus_id, 2)) STORED` column, same pattern as
`weekday_number`/`is_weekend` above: a pure function of an existing
column, so Postgres backfills and keeps it in sync automatically.

**Idempotent by construction**, so re-running just this cell or the
whole notebook end-to-end has the same effect: `ADD COLUMN IF NOT
EXISTS` for all three columns, and the `UPDATE` recomputes every row's
value fresh from `trip_fares`/`GARAGES` rather than incrementing
anything.

In [10]:
# (company_id, garage_lat, garage_lon). Hard-coded - see markdown above for
# why this can't be sourced from silver/bronze. Cross-checked against Google
# Maps; some companies have more than one garage on file, which is fine, the
# UPDATE below takes MIN() distance across every row sharing a company_id.
GARAGES = [
    ("02", -3.8028810192744027, -38.50341401619763),  # Auto Viação Fortaleza
    ("12", -3.7802332041766857, -38.5695361429177),  # Auto Viação São José
    ("14", -3.777246225871815, -38.568915329478166),  # Siará Grande
    ("20", -3.7215167402645686, -38.56100820100305),  # Santa Maria (1)
    ("20", -3.722886669415705, -38.556716242307836),  # Santa Maria (2)
    ("20", -3.7208313342318156, -38.56014978601272),  # Santa Maria (3)
    ("21", -3.704725956287296, -38.591196041413085),  # Transportes Urbanos Aliança
    ("26", -3.8332275953596895, -38.56380935858968),  # Maraponga Transportes
    ("30", -3.8072609664553823, -38.46878190037298),  # Viação Urbana
    ("35", -3.7134518990877323, -38.54376025481672),  # Vega (Jacarecanga)
    ("35", -3.8357607212799074, -38.49499371747953),  # Vega (Messejana)
    ("36", -3.746841903663143, -38.54357720067353),  # Santa Cecília (1)
    ("36", -3.7468204919231662, -38.54360938738769),  # Santa Cecília (2)
    ("42", -3.8035024356582543, -38.53854911444907),  # Auto Viação Dragão do Mar
    ("67", -3.7459634650123714, -38.598652172526734),  # COOTRAPS (Autran Nunes)
    ("67", -3.8424012552430677, -38.51473780117739),  # COOTRAPS (Jangurussu)
]

n_companies = len({company_id for company_id, _, _ in GARAGES})
print(f"{len(GARAGES)} garage locations across {n_companies} companies")

16 garage locations across 11 companies


In [11]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD COLUMN IF NOT EXISTS company_id TEXT
            GENERATED ALWAYS AS (LEFT(bus_id, 2)) STORED,
        ALTER COLUMN company_id SET NOT NULL;
""")
conn.execute(
    sql.SQL("COMMENT ON COLUMN ml.trip_validity_dataset.company_id IS {};").format(
        sql.Literal(
            "GENERATED ALWAYS AS (LEFT(bus_id, 2)) STORED. The 2-digit operator "
            "registry code the bus belongs to (e.g. '02' = Auto Viação Fortaleza, "
            "'67' = COOTRAPS) - leading zero significant, not a number. See "
            "GARAGES in this notebook for the code -> company mapping."
        )
    )
)
conn.commit()
print("company_id added")

company_id added


In [12]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD COLUMN IF NOT EXISTS trip_start_distance_to_nearest_garage_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_end_distance_to_nearest_garage_meters
            DOUBLE PRECISION;
""")

garage_values = sql.SQL(", ").join(
    sql.SQL("({}, {}, {})").format(
        sql.Literal(company_id), sql.Literal(lat), sql.Literal(lon)
    )
    for company_id, lat, lon in GARAGES
)

update_query = sql.SQL("""
    WITH garages (company_id, garage_lat, garage_lon) AS (
        VALUES {garage_values}
    ),
    -- one row per trip: its earliest/latest geo-tagged fare, never AVL
    first_geo_fare AS (
        SELECT DISTINCT ON (trip_id) trip_id, geom AS first_geom
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        ORDER BY trip_id, boarding_at ASC
    ),
    last_geo_fare AS (
        SELECT DISTINCT ON (trip_id) trip_id, geom AS last_geom
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        ORDER BY trip_id, boarding_at DESC
    ),
    distances AS (
        SELECT
            t.trip_id,
            MIN(ST_Distance(
                ff.first_geom::geography,
                ST_SetSRID(ST_MakePoint(g.garage_lon, g.garage_lat), 4326)::geography
            )) AS start_dist,
            MIN(ST_Distance(
                lf.last_geom::geography,
                ST_SetSRID(ST_MakePoint(g.garage_lon, g.garage_lat), 4326)::geography
            )) AS end_dist
        FROM ml.trip_validity_trips t
        LEFT JOIN first_geo_fare ff ON ff.trip_id = t.trip_id
        LEFT JOIN last_geo_fare lf ON lf.trip_id = t.trip_id
        JOIN garages g ON g.company_id = LEFT(t.bus_id, 2)
        WHERE ff.first_geom IS NOT NULL OR lf.last_geom IS NOT NULL
        GROUP BY t.trip_id
    )
    UPDATE ml.trip_validity_dataset d
    SET trip_start_distance_to_nearest_garage_meters = dist.start_dist,
        trip_end_distance_to_nearest_garage_meters = dist.end_dist
    FROM distances dist
    WHERE dist.trip_id = d.trip_id;
""").format(garage_values=garage_values)

conn.execute(update_query)

conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset."
        "trip_start_distance_to_nearest_garage_meters IS {};"
    ).format(
        sql.Literal(
            "ST_Distance (geography, meters) between this trip's FIRST "
            "geo-tagged AFC fare tap (ml.trip_validity_trip_fares, earliest "
            "boarding_at with geom IS NOT NULL - never an AVL position) and "
            "the closest entry in GARAGES (this notebook) sharing this trip's "
            "company_id. NULL when the trip has zero geo-tagged fares, or its "
            "bus_id's 2-digit prefix isn't in GARAGES."
        )
    )
)
conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset."
        "trip_end_distance_to_nearest_garage_meters IS {};"
    ).format(
        sql.Literal(
            "Same as trip_start_distance_to_nearest_garage_meters, but from "
            "this trip's LAST geo-tagged AFC fare tap (latest boarding_at "
            "with geom IS NOT NULL) instead of its first."
        )
    )
)
conn.commit()
print(
    "trip_start_distance_to_nearest_garage_meters, "
    "trip_end_distance_to_nearest_garage_meters populated"
)

trip_start_distance_to_nearest_garage_meters, trip_end_distance_to_nearest_garage_meters populated


In [13]:
EXPECTED_COLUMN_COUNT_AFTER_GARAGE_FEATURES = 85
EXPECTED_COMPANY_IDS = {c for c, _, _ in GARAGES}

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)
    if column_count != EXPECTED_COLUMN_COUNT_AFTER_GARAGE_FEATURES:
        msg = "column count mismatch after adding garage-distance features"
        raise AssertionError(msg)

    # company_id never falls outside the companies GARAGES actually covers
    cur.execute("SELECT DISTINCT company_id FROM ml.trip_validity_dataset;")
    seen_company_ids = {row[0] for row in cur.fetchall()}
    unknown_companies = seen_company_ids - EXPECTED_COMPANY_IDS
    print("distinct company_id values:", sorted(seen_company_ids))
    if unknown_companies:
        msg = f"company_id(s) with no entry in GARAGES: {unknown_companies}"
        raise AssertionError(msg)

    # both distance columns: never negative, never populated one-without-the-other
    cur.execute("""
        SELECT
            count(*) FILTER (WHERE trip_start_distance_to_nearest_garage_meters < 0
                              OR trip_end_distance_to_nearest_garage_meters < 0),
            count(*) FILTER (
                WHERE (trip_start_distance_to_nearest_garage_meters IS NULL)
                   != (trip_end_distance_to_nearest_garage_meters IS NULL)
            ),
            count(*) FILTER (WHERE trip_start_distance_to_nearest_garage_meters
                              IS NOT NULL)
        FROM ml.trip_validity_dataset;
    """)
    n_negative, n_mismatched_null, n_populated = cur.fetchone()
    print("rows with a negative garage distance:", n_negative)
    print("rows where only one of start/end is NULL:", n_mismatched_null)
    print("rows with a populated garage distance:", n_populated)
    if n_negative != 0:
        msg = "found a negative garage distance"
        raise AssertionError(msg)
    if n_mismatched_null != 0:
        msg = "start/end garage distance NULL-ness disagree for some rows"
        raise AssertionError(msg)

    cur.execute("""
        SELECT trip_id, bus_id, company_id,
               round(trip_start_distance_to_nearest_garage_meters::numeric, 1),
               round(trip_end_distance_to_nearest_garage_meters::numeric, 1)
        FROM ml.trip_validity_dataset
        WHERE trip_start_distance_to_nearest_garage_meters IS NOT NULL
        ORDER BY trip_id LIMIT 3;
    """)
    for row in cur.fetchall():
        print(row)

print("all checks passed")

columns: 85


distinct company_id values: ['02', '12', '14', '20', '21', '26', '30', '35', '36', '42', '67']
rows with a negative garage distance: 0
rows where only one of start/end is NULL: 0
rows with a populated garage distance: 899299
(1, '35128', '35', Decimal('1767.2'), Decimal('1491.9'))
(2, '20489', '20', Decimal('3469.1'), Decimal('3011.3'))
(3, '20597', '20', Decimal('3154.0'), Decimal('3154.0'))
all checks passed


## Adding route straight-line distance features

One more engineered feature: the straight-line ("as the crow flies")
distance between a route's own start and end point, for both the `-I`
and `-V` shapes - distinct from `route_i_length_meters`/
`route_v_length_meters`, which are the shape's actual road-following
length (`ST_Length`). Computed directly from `ml.trip_validity_route_shapes`
(built once already, in `02_gtfs_materialize.ipynb`) via
`ST_Distance(ST_StartPoint(shape_geom), ST_EndPoint(shape_geom))` - no
need to touch or re-run any of the earlier notebooks, this only reads
a table that already exists.

`route_i_straight_line_ratio`/`route_v_straight_line_ratio` (=
straight-line / actual length) come along too, same raw-then-ratio
convention as the rest of this dataset: close to `1` means a fairly
direct route, close to `0` means a winding or loop-shaped one. No
`GREATEST(0, ...)` clipping needed here (unlike `path_match_score_*`)
- straight-line distance between two points can never exceed the
length of *any* path connecting them (triangle inequality), so this
ratio is bounded to `[0, 1]` by construction, not just by convention;
the verification cell below checks that holds in practice too rather
than assuming it.

Per-route, not per-trip, so this is computed once per distinct
`(feed_version_date, shape_id)` in a CTE, then joined onto every trip
via the `gtfs_shape_id_i`/`gtfs_shape_id_v` this dataset already
carries - cheap regardless of table size. `NULL` for a direction with
no matched shape (same as `route_i_length_meters` already is), and
`ADD COLUMN IF NOT EXISTS` + an unconditional `UPDATE` keep this
idempotent like every other section here.

In [14]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD COLUMN IF NOT EXISTS route_i_straight_line_meters DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS route_i_straight_line_ratio DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS route_v_straight_line_meters DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS route_v_straight_line_ratio DOUBLE PRECISION;
""")

conn.execute("""
    WITH shape_straight_line AS (
        SELECT
            feed_version_date,
            shape_id,
            ST_Distance(
                ST_StartPoint(shape_geom)::geography,
                ST_EndPoint(shape_geom)::geography
            ) AS straight_line_meters
        FROM ml.trip_validity_route_shapes
    ),
    trip_shape_straight_lines AS (
        SELECT
            d.trip_id,
            si.straight_line_meters AS i_straight_line_meters,
            sv.straight_line_meters AS v_straight_line_meters
        FROM ml.trip_validity_dataset d
        LEFT JOIN shape_straight_line si
            ON si.feed_version_date = d.gtfs_feed_version_date
           AND si.shape_id = d.gtfs_shape_id_i
        LEFT JOIN shape_straight_line sv
            ON sv.feed_version_date = d.gtfs_feed_version_date
           AND sv.shape_id = d.gtfs_shape_id_v
    )
    UPDATE ml.trip_validity_dataset d
    SET route_i_straight_line_meters = t.i_straight_line_meters,
        route_i_straight_line_ratio =
            t.i_straight_line_meters / NULLIF(d.route_i_length_meters, 0),
        route_v_straight_line_meters = t.v_straight_line_meters,
        route_v_straight_line_ratio =
            t.v_straight_line_meters / NULLIF(d.route_v_length_meters, 0)
    FROM trip_shape_straight_lines t
    WHERE t.trip_id = d.trip_id;
""")

conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset.route_i_straight_line_meters IS {};"
    ).format(
        sql.Literal(
            "ST_Distance (geography, meters) between ST_StartPoint and "
            "ST_EndPoint of the matched -I shape's geometry "
            "(ml.trip_validity_route_shapes.shape_geom) - straight-line, "
            "not the road-following route_i_length_meters. NULL when no "
            "-I shape was matched (see gtfs_shape_id_i)."
        )
    )
)
conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset.route_i_straight_line_ratio IS {};"
    ).format(
        sql.Literal(
            "route_i_straight_line_meters / route_i_length_meters. Bounded "
            "[0, 1] by the triangle inequality, not by clipping: 1 = a "
            "perfectly direct route, near 0 = winding or loop-shaped. "
            "Ratio divides through NULLIF(denominator, 0), so a zero "
            "denominator gives NULL rather than the Infinity Postgres "
            "would otherwise silently produce."
        )
    )
)
conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset.route_v_straight_line_meters IS {};"
    ).format(sql.Literal("Same as route_i_straight_line_meters, direction V."))
)
conn.execute(
    sql.SQL(
        "COMMENT ON COLUMN ml.trip_validity_dataset.route_v_straight_line_ratio IS {};"
    ).format(
        sql.Literal(
            "Same as route_i_straight_line_ratio, direction V. Ratio "
            "divides through NULLIF(denominator, 0), so a zero "
            "denominator gives NULL rather than the Infinity Postgres "
            "would otherwise silently produce."
        )
    )
)
conn.commit()
print(
    "route_i_straight_line_meters, route_i_straight_line_ratio, "
    "route_v_straight_line_meters, route_v_straight_line_ratio populated"
)

route_i_straight_line_meters, route_i_straight_line_ratio, route_v_straight_line_meters, route_v_straight_line_ratio populated


In [15]:
EXPECTED_COLUMN_COUNT_AFTER_STRAIGHT_LINE = 89

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)
    if column_count != EXPECTED_COLUMN_COUNT_AFTER_STRAIGHT_LINE:
        msg = "column count mismatch after adding straight-line features"
        raise AssertionError(msg)

    # meters/ratio NULL-ness stays symmetric within each direction
    cur.execute("""
        SELECT
            count(*) FILTER (
                WHERE (route_i_straight_line_meters IS NULL)
                   != (route_i_straight_line_ratio IS NULL)
            ),
            count(*) FILTER (
                WHERE (route_v_straight_line_meters IS NULL)
                   != (route_v_straight_line_ratio IS NULL)
            )
        FROM ml.trip_validity_dataset;
    """)
    i_mismatch, v_mismatch = cur.fetchone()
    print("rows where i meters/ratio NULL-ness disagree:", i_mismatch)
    print("rows where v meters/ratio NULL-ness disagree:", v_mismatch)
    if i_mismatch != 0 or v_mismatch != 0:
        msg = "straight-line meters/ratio NULL-ness disagree for some rows"
        raise AssertionError(msg)

    # the triangle-inequality bound actually holds, not just assumed
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE (route_i_straight_line_ratio IS NOT NULL
               AND (route_i_straight_line_ratio < 0
                    OR route_i_straight_line_ratio > 1))
           OR (route_v_straight_line_ratio IS NOT NULL
               AND (route_v_straight_line_ratio < 0
                    OR route_v_straight_line_ratio > 1));
    """)
    out_of_range = cur.fetchone()[0]
    print("straight_line_ratio rows out of [0,1]:", out_of_range)
    if out_of_range != 0:
        msg = "straight_line_ratio found out of [0,1] range"
        raise AssertionError(msg)

    cur.execute("""
        SELECT trip_id, gtfs_shape_id_i, gtfs_shape_id_v,
               round(route_i_straight_line_meters::numeric, 1),
               round(route_i_length_meters::numeric, 1),
               round(route_i_straight_line_ratio::numeric, 3)
        FROM ml.trip_validity_dataset
        WHERE route_i_straight_line_ratio IS NOT NULL
        ORDER BY trip_id LIMIT 3;
    """)
    for row in cur.fetchall():
        print(row)

print("all checks passed")

columns: 89


rows where i meters/ratio NULL-ness disagree: 0
rows where v meters/ratio NULL-ness disagree: 0
straight_line_ratio rows out of [0,1]: 0
(1, 'shape0102-I', 'shape0102-V', Decimal('4693.4'), Decimal('5644.6'), Decimal('0.831'))
(2, 'shape0101-I', 'shape0101-V', Decimal('7109.6'), Decimal('9326.1'), Decimal('0.762'))
(3, 'shape0225-I', 'shape0225-V', Decimal('3330.0'), Decimal('4818.9'), Decimal('0.691'))
all checks passed


## Adding terminal-distance features

Six more features: distance from the trip's first and last geo-tagged
AFC fare tap (again, never AVL) to the nearest bus terminal - three
variants each (any terminal, nearest *open* terminal, nearest *closed*
terminal), same "first/last fare, never AVL" basis as the garage
distances above, just against a different reference set and without a
company restriction (terminals are shared city infrastructure, not
tied to one operator, so every trip compares against all of them, not
just its own company's).

Terminal coordinates are **hard-coded** below (`TERMINALS`), this time
sourced directly from ETUFOR's (Empresa de Transporte Urbano de
Fortaleza) own published terminal registry (2024), not manually pinned
- the data came with lat/lon and a `tipologia` field already attached.
`tipologia` is `"Terminal Fechado"` (closed - an enclosed, gated
facility) or `"Terminal Aberto"` (open - an unenclosed plaza/praça);
mapped here to a plain `is_closed` boolean. Same caveat as `GARAGES`:
no upstream feed, so a new/closed terminal needs a manual edit here.

Structurally the same pattern as the garage-distance UPDATE:
`first_geo_fare`/`last_geo_fare` CTEs (identical definitions - not
touching `trip_path`, for the same single-tap-trip reason as before),
cross-joined against `TERMINALS` this time (no per-company join key),
with the open/closed variants computed via `MIN(...) FILTER (WHERE
...)` in the same aggregation pass as the unconditional minimum rather
than three separate scans. `NULL` when the trip has zero geo-tagged
fares, same as every other fare-tap-based feature here. Idempotent by
the same `ADD COLUMN IF NOT EXISTS` + unconditional `UPDATE`
construction.

In [16]:
# (name, terminal_lat, terminal_lon, is_closed). Hard-coded from ETUFOR's
# own 2024 terminal registry (came with lat/lon + tipologia attached, not
# manually pinned like GARAGES). "Terminal Fechado" (closed, an enclosed
# gated facility) -> True, "Terminal Aberto" (open plaza) -> False.
TERMINALS = [
    ("Conjunto Ceará", -3.772986, -38.607545, True),
    ("Papicu", -3.738568, -38.484761, True),
    ("Parangaba", -3.775878, -38.56358, True),
    ("Antonio Bezerra", -3.73752, -38.584548, True),
    ("Siqueira", -3.789302, -38.586853, True),
    ("Lagoa", -3.771151, -38.569879, True),
    ("Messejana", -3.831156, -38.502458, True),
    ("Washington Soares", -3.81501801194818, -38.4808807997053, False),
    ("Sagrado Coração de Jesus", -3.732300515031093, -38.52671674368621, False),
    ("Jose Walter", -3.8222248574044375, -38.55851841148358, False),
    ("José de Alencar", -3.726297474312544, -38.53210776782962, False),
]

n_open = sum(1 for _, _, _, closed in TERMINALS if not closed)
n_closed = sum(1 for _, _, _, closed in TERMINALS if closed)
print(f"{len(TERMINALS)} terminals ({n_open} open, {n_closed} closed)")

11 terminals (4 open, 7 closed)


In [17]:
conn.execute("""
    ALTER TABLE ml.trip_validity_dataset
        ADD COLUMN IF NOT EXISTS trip_start_distance_to_nearest_terminal_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_start_distance_to_nearest_open_terminal_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_start_distance_to_nearest_closed_terminal_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_end_distance_to_nearest_terminal_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_end_distance_to_nearest_open_terminal_meters
            DOUBLE PRECISION,
        ADD COLUMN IF NOT EXISTS trip_end_distance_to_nearest_closed_terminal_meters
            DOUBLE PRECISION;
""")

terminal_values = sql.SQL(", ").join(
    sql.SQL("({}, {}, {}, {})").format(
        sql.Literal(name), sql.Literal(lat), sql.Literal(lon), sql.Literal(is_closed)
    )
    for name, lat, lon, is_closed in TERMINALS
)

update_query = sql.SQL("""
    WITH terminals (name, terminal_lat, terminal_lon, is_closed) AS (
        VALUES {terminal_values}
    ),
    first_geo_fare AS (
        SELECT DISTINCT ON (trip_id) trip_id, geom AS first_geom
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        ORDER BY trip_id, boarding_at ASC
    ),
    last_geo_fare AS (
        SELECT DISTINCT ON (trip_id) trip_id, geom AS last_geom
        FROM ml.trip_validity_trip_fares
        WHERE geom IS NOT NULL
        ORDER BY trip_id, boarding_at DESC
    ),
    distances AS (
        SELECT
            t.trip_id,
            MIN(ST_Distance(ff.first_geom::geography, term_point.geog))
                AS start_dist_any,
            MIN(ST_Distance(ff.first_geom::geography, term_point.geog))
                FILTER (WHERE NOT term.is_closed) AS start_dist_open,
            MIN(ST_Distance(ff.first_geom::geography, term_point.geog))
                FILTER (WHERE term.is_closed) AS start_dist_closed,
            MIN(ST_Distance(lf.last_geom::geography, term_point.geog))
                AS end_dist_any,
            MIN(ST_Distance(lf.last_geom::geography, term_point.geog))
                FILTER (WHERE NOT term.is_closed) AS end_dist_open,
            MIN(ST_Distance(lf.last_geom::geography, term_point.geog))
                FILTER (WHERE term.is_closed) AS end_dist_closed
        FROM ml.trip_validity_trips t
        LEFT JOIN first_geo_fare ff ON ff.trip_id = t.trip_id
        LEFT JOIN last_geo_fare lf ON lf.trip_id = t.trip_id
        CROSS JOIN terminals term
        CROSS JOIN LATERAL (
            SELECT ST_SetSRID(
                ST_MakePoint(term.terminal_lon, term.terminal_lat), 4326
            )::geography AS geog
        ) term_point
        WHERE ff.first_geom IS NOT NULL OR lf.last_geom IS NOT NULL
        GROUP BY t.trip_id
    )
    UPDATE ml.trip_validity_dataset d
    SET trip_start_distance_to_nearest_terminal_meters = dist.start_dist_any,
        trip_start_distance_to_nearest_open_terminal_meters = dist.start_dist_open,
        trip_start_distance_to_nearest_closed_terminal_meters = dist.start_dist_closed,
        trip_end_distance_to_nearest_terminal_meters = dist.end_dist_any,
        trip_end_distance_to_nearest_open_terminal_meters = dist.end_dist_open,
        trip_end_distance_to_nearest_closed_terminal_meters = dist.end_dist_closed
    FROM distances dist
    WHERE dist.trip_id = d.trip_id;
""").format(terminal_values=terminal_values)

conn.execute(update_query)

terminal_col_comments = [
    (
        "trip_start_distance_to_nearest_terminal_meters",
        "ST_Distance (geography, meters) between this trip's FIRST "
        "geo-tagged AFC fare tap (ml.trip_validity_trip_fares, earliest "
        "boarding_at with geom IS NOT NULL - never an AVL position) and "
        "the closest entry in TERMINALS (this notebook), any tipologia. "
        "NULL when the trip has zero geo-tagged fares.",
    ),
    (
        "trip_start_distance_to_nearest_open_terminal_meters",
        "Same as trip_start_distance_to_nearest_terminal_meters, "
        "restricted to TERMINALS entries with is_closed = false "
        "('Terminal Aberto'). NULL if the trip has zero geo-tagged fares.",
    ),
    (
        "trip_start_distance_to_nearest_closed_terminal_meters",
        "Same as trip_start_distance_to_nearest_terminal_meters, "
        "restricted to TERMINALS entries with is_closed = true "
        "('Terminal Fechado'). NULL if the trip has zero geo-tagged fares.",
    ),
    (
        "trip_end_distance_to_nearest_terminal_meters",
        "Same as trip_start_distance_to_nearest_terminal_meters, but from "
        "this trip's LAST geo-tagged AFC fare tap (latest boarding_at "
        "with geom IS NOT NULL) instead of its first.",
    ),
    (
        "trip_end_distance_to_nearest_open_terminal_meters",
        "Same as trip_start_distance_to_nearest_open_terminal_meters, but "
        "from this trip's LAST geo-tagged AFC fare tap instead of its "
        "first.",
    ),
    (
        "trip_end_distance_to_nearest_closed_terminal_meters",
        "Same as trip_start_distance_to_nearest_closed_terminal_meters, "
        "but from this trip's LAST geo-tagged AFC fare tap instead of its "
        "first.",
    ),
]
for col, text in terminal_col_comments:
    conn.execute(
        sql.SQL("COMMENT ON COLUMN ml.trip_validity_dataset.{} IS {};").format(
            sql.Identifier(col), sql.Literal(text)
        )
    )

conn.commit()
print("trip_start/end_distance_to_nearest_[open/closed_]terminal_meters populated")

trip_start/end_distance_to_nearest_[open/closed_]terminal_meters populated


In [18]:
EXPECTED_COLUMN_COUNT_AFTER_TERMINAL_FEATURES = 95
EXPECTED_TERMINAL_COUNT = 11

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*) FROM information_schema.columns
        WHERE table_schema = 'ml' AND table_name = 'trip_validity_dataset';
    """)
    column_count = cur.fetchone()[0]
    print("columns:", column_count)
    if column_count != EXPECTED_COLUMN_COUNT_AFTER_TERMINAL_FEATURES:
        msg = "column count mismatch after adding terminal-distance features"
        raise AssertionError(msg)

    if len(TERMINALS) != EXPECTED_TERMINAL_COUNT:
        msg = f"expected {EXPECTED_TERMINAL_COUNT} terminals, found {len(TERMINALS)}"
        raise AssertionError(msg)

    # a restricted (open-only/closed-only) minimum can never beat the
    # unrestricted "any terminal" minimum - it's a minimum over a subset
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE trip_start_distance_to_nearest_open_terminal_meters
                  < trip_start_distance_to_nearest_terminal_meters
           OR trip_start_distance_to_nearest_closed_terminal_meters
                  < trip_start_distance_to_nearest_terminal_meters
           OR trip_end_distance_to_nearest_open_terminal_meters
                  < trip_end_distance_to_nearest_terminal_meters
           OR trip_end_distance_to_nearest_closed_terminal_meters
                  < trip_end_distance_to_nearest_terminal_meters;
    """)
    impossible = cur.fetchone()[0]
    print(
        "rows where a restricted-subset minimum beats the overall minimum:",
        impossible,
    )
    if impossible != 0:
        msg = "an open/closed-restricted distance is smaller than the overall minimum"
        raise AssertionError(msg)

    # any/open/closed all come from the same distances CTE row, so their
    # NULL-ness must agree (all populated or all NULL) - TERMINALS has at
    # least one open and one closed entry, so this isn't coincidental
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_dataset
        WHERE (trip_start_distance_to_nearest_terminal_meters IS NULL)
                  != (trip_start_distance_to_nearest_open_terminal_meters IS NULL)
           OR (trip_start_distance_to_nearest_terminal_meters IS NULL)
                  != (trip_start_distance_to_nearest_closed_terminal_meters IS NULL)
           OR (trip_end_distance_to_nearest_terminal_meters IS NULL)
                  != (trip_end_distance_to_nearest_open_terminal_meters IS NULL)
           OR (trip_end_distance_to_nearest_terminal_meters IS NULL)
                  != (trip_end_distance_to_nearest_closed_terminal_meters IS NULL);
    """)
    null_mismatch = cur.fetchone()[0]
    print("rows where any/open/closed NULL-ness disagree:", null_mismatch)
    if null_mismatch != 0:
        msg = "any/open/closed terminal distance NULL-ness disagree"
        raise AssertionError(msg)

    cur.execute("""
        SELECT count(*) FILTER (
            WHERE trip_start_distance_to_nearest_terminal_meters < 0
               OR trip_end_distance_to_nearest_terminal_meters < 0
        )
        FROM ml.trip_validity_dataset;
    """)
    n_negative = cur.fetchone()[0]
    print("rows with a negative terminal distance:", n_negative)
    if n_negative != 0:
        msg = "found a negative terminal distance"
        raise AssertionError(msg)

    cur.execute("""
        SELECT trip_id,
               round(trip_start_distance_to_nearest_terminal_meters::numeric, 1),
               round(trip_start_distance_to_nearest_open_terminal_meters::numeric, 1),
               round(trip_start_distance_to_nearest_closed_terminal_meters::numeric, 1)
        FROM ml.trip_validity_dataset
        WHERE trip_start_distance_to_nearest_terminal_meters IS NOT NULL
        ORDER BY trip_id LIMIT 3;
    """)
    for row in cur.fetchall():
        print(row)

print("all checks passed")

columns: 95


rows where a restricted-subset minimum beats the overall minimum: 0


rows where any/open/closed NULL-ness disagree: 0


rows with a negative terminal distance: 0
(1, Decimal('434.9'), Decimal('434.9'), Decimal('5402.3'))
(2, Decimal('3170.7'), Decimal('6676.3'), Decimal('3170.7'))
(3, Decimal('62.7'), Decimal('5976.1'), Decimal('62.7'))
all checks passed


## Refresh the table-level comment

`COMMENT ON TABLE` was set once, right after the original 80-column
build, and never updated as `weekday_number`/`is_weekend`, then
`company_id`/the garage-distance columns, then the route
straight-line columns, then the terminal-distance columns were added
afterward - overwriting it here so `\d+ ml.trip_validity_dataset`
reflects the table's actual final shape instead of a stale "80
columns".

In [19]:
conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_dataset IS {};").format(
        sql.Literal(
            "Trip Validity model: the final ML-ready dataset. One row per "
            "trip, 95 columns - the original 80 (identifiers + raw metrics "
            "from ml.trip_validity_trip_metrics + every relative/normalized "
            "column) plus weekday_number/is_weekend, company_id/"
            "trip_start_distance_to_nearest_garage_meters/"
            "trip_end_distance_to_nearest_garage_meters, "
            "route_i_straight_line_meters/route_i_straight_line_ratio/"
            "route_v_straight_line_meters/route_v_straight_line_ratio, and "
            "trip_start/end_distance_to_nearest_[open/closed_]terminal_meters "
            "(6 columns), all added afterward. See "
            "ml/trip_validity_model/notebooks/05_final_dataset.ipynb."
        )
    )
)
conn.commit()
print("table comment refreshed")

table comment refreshed
